# Human Activity Intensity (HAI) – Russia–China border, 2016–2026
Re-creation of Cheng, Liu, Chen & Liu (2022), *Human Activity Intensity and Its Spatial-Temporal Evolution in China's Border Areas*, Land 11(7):1089, restricted to the China–Russia border and run as an annual series.

**Method (paper §2.2–2.3):** 50 km buffer either side of the border. HAI = (land + pop + ntl) / 3 on a 0–10 scale, where
* **land** – land-cover score (Table 1: built-up 10, cropland 6, grassland 2, water 1, other 0)
* **pop** – population density ÷ the maximum over all years × 10 (eq. 1)
* **ntl** – night-time light ÷ the maximum over all years × 10 (eq. 2)

Classes (Table 2): >4 high, 3–4 relatively high, 1–3 relatively low, 0.1–1 low, <0.1 very low.

**Focus:** where the Chinese and Russian sides differ most (per-cell contrast, hotspot type, and how that contrast moved 2016 → 2026).

**Deviations from the paper / caveats**
* Land cover is Google **Dynamic World** (annual May–Sep mode, 10 m) instead of ESA CCI 300 m, because CCI is not in Earth Engine and WorldCover only has 2020/2021. Dynamic World cannot separate rainfed from irrigated cropland, so all cropland scores 6.
* Night lights are VIIRS VNP46A2 (May–Sep median, avoids snow) rather than the harmonised DMSP-VIIRS product. Summer twilight at ~50°N can lift radiance a little.
* GHS-POP only has 2015 / 2020 / 2025 epochs: intermediate years are linearly interpolated and 2026 re-uses 2025.
* 2026 nightlights/land cover run to the end of September 2026 (the May–Sep window is complete).
* The paper's max-normalisation means a few bright cities set the scale, so most of the border sits near zero. This is intended (the paper reports 93% of cells below 1).
* Side split uses the LSIB country polygons, which can differ slightly from the supplied border line.

In [ ]:
%pip install earthengine-api geemap
import ee
import geemap
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

In [ ]:
ee.Authenticate()
ee.Initialize(project='editorial-democraticprimaries')

In [ ]:
# --- Parameters ---------------------------------------------------------
YEARS      = list(range(2016, 2027))
FIRST, LAST = YEARS[0], YEARS[-1]
BUFFER_M   = 50000        # border buffer each side (m), as in the paper
GRID_M     = 1000         # common grid (m), as in the paper
SEASON     = (5, 9)       # May-Sep composite (avoids snow)
HIGH_HAI   = 4            # 'high' class threshold (Table 2)
CELL_M     = 20000        # contrast-analysis cell size (approx, in degrees at the equator)
MIN_PIX    = 40           # min 1 km pixels on EACH side for a cell to count as 'cross-border'
HI_MIN_KM2 = 3            # min km2 of HAI>4 on a side to call it a hotspot
POP_EPOCHS = (2015, 2020, 2025)

# Dynamic World label index -> Table 1 score (0-10); everything unlisted scores 0
#   0 water, 2 grass, 3 flooded veg, 4 crops, 6 built
LC_SCORE = {0: 1, 2: 2, 3: 1, 4: 6, 6: 10}

# set after the Export cell below has finished, to skip the heavy recompute
ASSET_ID = None  # e.g. 'projects/editorial-democraticprimaries/assets/hai_ru_cn_2016_2026'

In [ ]:
# --- Geometry, grid and country sides ----------------------------------------
border = geemap.geojson_to_ee('china-russia-border.geojson')
borderGeom = border.geometry()
buf = borderGeom.buffer(BUFFER_M)

GRID = ee.Projection('EPSG:4326').atScale(GRID_M)

lsib = ee.FeatureCollection('USDOS/LSIB_SIMPLE/2017')
cnMask = ee.Image(1).clip(lsib.filter(ee.Filter.eq('country_na', 'China')).geometry())
ruMask = ee.Image(1).clip(lsib.filter(ee.Filter.eq('country_na', 'Russia')).geometry())
print('LSIB China / Russia features:',
      lsib.filter(ee.Filter.eq('country_na', 'China')).size().getInfo(),
      lsib.filter(ee.Filter.eq('country_na', 'Russia')).size().getInfo())

def to_grid(img, max_pixels=65536):
    # aggregate a finer image onto the common grid (mean)
    return (img.reduceResolution(reducer=ee.Reducer.mean(), maxPixels=max_pixels)
               .reproject(GRID))

In [ ]:
# --- Raw layers --------------------------------------------------------------
DW = ee.ImageCollection('GOOGLE/DYNAMICWORLD/V1')
DW_PROJ = ee.Projection('EPSG:4326').atScale(10)

def land_score(year):
    # mean Table-1 score (0-10) of the 10 m cells inside each grid cell
    label = (DW.filterBounds(buf)
               .filterDate(f'{year}-{SEASON[0]:02d}-01', f'{year}-{SEASON[1] + 1:02d}-01')
               .select('label').mode().setDefaultProjection(DW_PROJ))
    score = (ee.Image.cat([label.eq(c).multiply(v) for c, v in LC_SCORE.items()])
               .reduce(ee.Reducer.sum()))
    return to_grid(score)          # eq()-sum scoring, not remap(): avoids the 2^31 pixel limit

GHS = ee.ImageCollection('JRC/GHSL/P2023A/GHS_POP')
popEpoch = {e: to_grid(GHS.filter(ee.Filter.eq('system:index', str(e))).first()
                          .select('population_count').multiply(100))   # 100 m cell -> people/km2
            for e in POP_EPOCHS}

def pop_density(year):
    lo = max(e for e in POP_EPOCHS if e <= year)
    hi = min([e for e in POP_EPOCHS if e >= year], default=POP_EPOCHS[-1])
    if lo == hi:
        return popEpoch[lo]
    w = (year - lo) / (hi - lo)
    return popEpoch[lo].multiply(1 - w).add(popEpoch[hi].multiply(w))

VIIRS = (ee.ImageCollection('NASA/VIIRS/002/VNP46A2')
           .select('Gap_Filled_DNB_BRDF_Corrected_NTL'))
VIIRS_PROJ = VIIRS.first().projection()

def ntl_raw(year):
    img = (VIIRS.filter(ee.Filter.calendarRange(year, year, 'year'))
                .filter(ee.Filter.calendarRange(SEASON[0], SEASON[1], 'month'))
                .median().multiply(0.1)          # 0.1 scale factor -> nW/cm2/sr
                .setDefaultProjection(VIIRS_PROJ))
    return to_grid(img)

# which nightlights nights went into the latest composite?
_v = (VIIRS.filter(ee.Filter.calendarRange(LAST, LAST, 'year'))
           .filter(ee.Filter.calendarRange(SEASON[0], SEASON[1], 'month')))
print(f'{LAST} nightlights nights used:', _v.size().getInfo(),
      '| latest:', ee.Date(_v.aggregate_max('system:time_start')).format('YYYY-MM-dd').getInfo())

In [38]:
# --- HAI stack: bands land_Y, pop_Y, ntl_Y, hai_Y for every year (0-10 scale) -----
BANDS = ['land', 'pop', 'ntl', 'hai']
LAND_ASSET = 'projects/editorial-democraticprimaries/assets/hai_land_{}'

def asset_exists(a):
    try:
        ee.data.getAsset(a); return True
    except ee.EEException:
        return False

# Step 1: the heavy part (10 m Dynamic World) is cached ONE YEAR AT A TIME as a 1 km asset.
todo = [y for y in YEARS if not asset_exists(LAND_ASSET.format(y))]
for y in todo:
    t = ee.batch.Export.image.toAsset(
        image=land_score(y).rename('land').float().clip(buf),
        description=f'hai_land_{y}', assetId=LAND_ASSET.format(y),
        region=buf, scale=GRID_M, crs='EPSG:4326', maxPixels=1e13)
    t.start()
    print('started land export', y, t.id)

if todo:
    print(f'\n{len(todo)} land exports running -- wait for them to finish (Tasks tab), then re-run this cell.')
    stack = None
else:
    # Step 2: everything left is cheap (VIIRS + GHSL + cached land)
    raw = {y: {'land': ee.Image(LAND_ASSET.format(y)).select('land'),
               'pop': pop_density(y), 'ntl': ntl_raw(y)} for y in YEARS}

    def over_years(key):
        return ee.ImageCollection([raw[y][key].unmask(0) for y in YEARS]).max()
    maxImg = ee.Image.cat([over_years('pop').rename('pop'), over_years('ntl').rename('ntl')])
    kw = dict(geometry=buf, scale=GRID_M, maxPixels=1e13, tileScale=16)
    maxima = maxImg.reduceRegion(ee.Reducer.max(), **kw).getInfo()
    print('max over years:', maxima)

    def build_year(y):
        land = raw[y]['land'].unmask(0)
        pop  = raw[y]['pop'].unmask(0).divide(maxima['pop']).multiply(10)
        ntl  = raw[y]['ntl'].unmask(0).divide(maxima['ntl']).multiply(10)
        hai  = land.add(pop).add(ntl).divide(3)                     # eq. 3
        return ee.Image.cat([land, pop, ntl, hai]).rename([f'{b}_{y}' for b in BANDS])

    stack = ee.Image.cat([build_year(y) for y in YEARS]).float().clip(buf)

started land export 2016 HKC66SOWZT22YODBN6EH2RZZ
started land export 2017 BWA3QMIDRVUQK2CI3RDDVSLZ
started land export 2018 UEWDPQ76CPA3BJQPLDUMK4XJ
started land export 2019 Z57G3TG5PQKCIHT4QEDQP5UK
started land export 2020 ZN5SJDETQ2LODRCNZRPUEELT
started land export 2021 MBVZBOBM4OCSHAUSJB3LB6UA
started land export 2022 J2Q4237FY6EW2CRX4FT7NZ7S
started land export 2023 ISCPVVB7VBAHTP2YQ7EPMYU4
started land export 2024 Y2QKK4ZPICHJYGDQSX77IGTU
started land export 2025 CUMKEKRBGQ7WABKW252YWW5R
started land export 2026 C24T3DJWB2YZQAEW54SYVK2K

11 land exports running -- wait for them to finish (Tasks tab), then re-run this cell.


In [ ]:
# --- Optional: export the stack once, then set ASSET_ID above and re-run -------
# Dynamic World at 10 m over a ~400,000 km2 buffer for 11 years is heavy; caching it
# as a 1 km asset makes every later cell fast.
RUN_EXPORT = False
if RUN_EXPORT and not ASSET_ID:
    task = ee.batch.Export.image.toAsset(
        image=stack, description='hai_ru_cn_2016_2026',
        assetId='projects/editorial-democraticprimaries/assets/hai_ru_cn_2016_2026',
        region=buf, scale=GRID_M, crs='EPSG:4326', maxPixels=1e13)
    task.start()
    print('export started:', task.id)

In [ ]:
# --- Maps: HAI, change and classes ----------------------------------------------
haiVis    = {'min': 0, 'max': 4, 'palette': ['#000004', '#51127c', '#b73779', '#fc8961', '#fcfdbf']}
changeVis = {'min': -2, 'max': 2, 'palette': ['#b2182b', '#f7f7f7', '#2166ac']}
classVis  = {'min': 1, 'max': 5, 'palette': ['#2c0b54', '#51127c', '#b73779', '#fc8961', '#fcfdbf']}

def hai(y):
    return stack.select(f'hai_{y}')

def classify(h):
    # Table 2: 1 very low (<0.1), 2 low, 3 relatively low, 4 relatively high, 5 high (>4)
    return (ee.Image(1).where(h.gte(0.1), 2).where(h.gte(1), 3)
            .where(h.gte(3), 4).where(h.gt(4), 5).updateMask(h.mask()).rename('class'))

def base_map(center=(50, 128), zoom=5):
    m = geemap.Map(center=list(center), zoom=zoom)
    m.clear_layers()
    m.add_tile_layer('https://mt1.google.com/vt/lyrs=m&x={x}&y={y}&z={z}&hl=en',
                     name='Google Roads (EN)', attribution='Google')
    return m

borderLine = ee.Image().byte().paint(ee.FeatureCollection([ee.Feature(borderGeom)]), 1, 2)

m = base_map()
m.addLayer(hai(LAST), haiVis, f'HAI {LAST} (0-10, stretched to 4)')
m.addLayer(hai(FIRST), haiVis, f'HAI {FIRST}', False)
m.addLayer(hai(LAST).subtract(hai(FIRST)), changeVis, f'HAI change {FIRST} -> {LAST}', False)
m.addLayer(classify(hai(FIRST)), classVis, f'HAI class {FIRST} (1 very low - 5 high)', False)
m.addLayer(classify(hai(LAST)), classVis, f'HAI class {LAST}', False)
for b in ['land', 'pop', 'ntl']:
    m.addLayer(stack.select(f'{b}_{LAST}'), {'min': 0, 'max': 10, 'palette': haiVis['palette']},
               f'{b} {LAST} (0-10)', False)
m.addLayer(ee.Image().byte().paint(ee.FeatureCollection([ee.Feature(buf)]), 1, 2),
           {'palette': 'white'}, 'Border buffer', False)
m.addLayer(borderLine, {'palette': 'cyan'}, 'Border')
m

In [37]:
# --- Side-by-side comparison (paper Fig. 5 analogue) ---------------------------------
sideImg = ee.Image.cat(
    [hai(y).updateMask(cnMask).rename(f'cn_{y}') for y in YEARS] +
    [hai(y).updateMask(ruMask).rename(f'ru_{y}') for y in YEARS])

# one small request per year (22 bands at once exceeds EE's reprojection limit)
sideMeans = {}
for y in YEARS:
    r = sideImg.select([f'cn_{y}', f'ru_{y}']).reduceRegion(
        ee.Reducer.mean(), buf, crs=GRID, maxPixels=1e13, tileScale=16).getInfo()
    sideMeans.update(r)

ts = pd.DataFrame({'China':  [sideMeans[f'cn_{y}'] for y in YEARS],
                   'Russia': [sideMeans[f'ru_{y}'] for y in YEARS]}, index=YEARS)
ts['China - Russia'] = ts['China'] - ts['Russia']

ax = ts[['China', 'Russia']].plot(marker='o', figsize=(7, 4), color=['#d62728', '#1f77b4'])
ax.set_ylabel('Mean HAI (0-10), 50 km buffer')
ax.set_title('Russia–China border: mean HAI by side')
ax.grid(alpha=.3)
plt.show()

print(f'Mean change {FIRST} -> {LAST}:  China {ts.China.iloc[-1] - ts.China.iloc[0]:+.3f}   '
      f'Russia {ts.Russia.iloc[-1] - ts.Russia.iloc[0]:+.3f}')
ts.round(3)

KeyboardInterrupt: 

In [ ]:
# --- Cross-border contrast per ~20 km cell ---------------------------------------------
cells = buf.coveringGrid(ee.Projection('EPSG:4326').atScale(CELL_M))

# 1) pixel counts per side, and km2 of HAI>4 per side in the first/last year
inBuf = hai(FIRST).mask()
countImg = ee.Image.cat(
    [cnMask.updateMask(inBuf).rename('n_cn'), ruMask.updateMask(inBuf).rename('n_ru')] +
    [hai(y).gt(HIGH_HAI).updateMask(cnMask).rename(f'hi_cn_{y}') for y in (FIRST, LAST)] +
    [hai(y).gt(HIGH_HAI).updateMask(ruMask).rename(f'hi_ru_{y}') for y in (FIRST, LAST)])
cells = countImg.reduceRegions(cells, ee.Reducer.sum(), scale=GRID_M, tileScale=4)

# keep only cells that really straddle the border
cells = cells.filter(ee.Filter.And(ee.Filter.gte('n_cn', MIN_PIX), ee.Filter.gte('n_ru', MIN_PIX)))

# 2) mean HAI per side, per year
cells = sideImg.reduceRegions(cells, ee.Reducer.mean(), scale=GRID_M, tileScale=4)

def add_contrast(f):
    d = {f'd_{y}': ee.Number(f.get(f'cn_{y}')).subtract(ee.Number(f.get(f'ru_{y}'))) for y in YEARS}
    c = f.geometry().centroid(100).coordinates()
    return f.set(d).set({'lon': c.get(0), 'lat': c.get(1)})

cells = cells.map(add_contrast)

df = pd.DataFrame([f['properties'] for f in cells.getInfo()['features']])
print(len(df), 'cross-border cells')

In [ ]:
# --- Rank contrast, label nearest town, classify hotspot type -------------------------
import math

PLACES = {  # approximate lon, lat – used only to label the nearest known twin town (my own list)
    'Heihe / Blagoveshchensk': (127.52, 50.26), 'Manzhouli / Zabaykalsk': (117.35, 49.62),
    'Suifenhe / Pogranichny': (131.25, 44.40), 'Khabarovsk / Fuyuan': (135.07, 48.48),
    'Hunchun / Khasan': (130.37, 42.87), 'Mohe': (122.52, 52.97),
    'Tongjiang / Nizhneleninskoye': (132.51, 47.64), 'Dongning / Poltavka': (131.12, 44.07),
    'Raohe / Bikin': (134.02, 46.80), 'Jiayin / Birobidzhan': (130.40, 48.89),
    'Luobei / Amur-Zeya': (130.60, 47.58), 'Huma / Skovorodino': (126.66, 51.72),
    'Hulunbuir / Dauria': (117.60, 49.50), 'Ussuriysk': (131.88, 43.80),
}

def km(lon1, lat1, lon2, lat2):
    p = math.pi / 180
    a = (math.sin((lat2 - lat1) * p / 2) ** 2 +
         math.cos(lat1 * p) * math.cos(lat2 * p) * math.sin((lon2 - lon1) * p / 2) ** 2)
    return 12742 * math.asin(math.sqrt(a))

def nearest(lon, lat, within=60):
    name, d = min(((n, km(lon, lat, *xy)) for n, xy in PLACES.items()), key=lambda t: t[1])
    return name if d <= within else '-'

def hotspot_type(cn_hi, ru_hi):
    cn, ru = cn_hi >= HI_MIN_KM2, ru_hi >= HI_MIN_KM2
    if cn and ru: return 'both sides (bilateral / twin city)'
    if cn:        return 'China only (unilateral)'
    if ru:        return 'Russia only (unilateral)'
    return 'no HAI>4 hotspot'

df['near'] = [nearest(a, b) for a, b in zip(df.lon, df.lat)]
for y in (FIRST, LAST):
    df[f'type_{y}'] = [hotspot_type(a, b) for a, b in zip(df[f'hi_cn_{y}'], df[f'hi_ru_{y}'])]
df['abs_d_last'] = df[f'd_{LAST}'].abs()
df['d_change']   = df[f'd_{LAST}'] - df[f'd_{FIRST}']

cols = ['near', 'lon', 'lat', f'cn_{FIRST}', f'ru_{FIRST}', f'cn_{LAST}', f'ru_{LAST}',
        f'd_{FIRST}', f'd_{LAST}', 'd_change', f'type_{LAST}']
fmt = lambda t: t.round(2).reset_index(drop=True)

print(f'Top 15 cells by |China - Russia| HAI contrast in {LAST}  (nearest-town label is approximate)')
top = df.sort_values('abs_d_last', ascending=False).head(15)
display(fmt(top[cols]))

print(f'Where the contrast moved most {FIRST} -> {LAST}  (positive = China pulled ahead)')
display(fmt(df.reindex(df.d_change.abs().sort_values(ascending=False).index).head(10)[cols]))

print('Hotspot type counts')
display(pd.crosstab(df[f'type_{FIRST}'], df[f'type_{LAST}'], margins=True))

In [ ]:
# --- Contrast map -------------------------------------------------------------------------------
contrastVis = {'min': -3, 'max': 3, 'palette': ['#2166ac', '#f7f7f7', '#b2182b']}   # red = China higher
paint = lambda prop: ee.Image().float().paint(cells, prop)

m2 = base_map()
m2.addLayer(paint(f'd_{LAST}'), contrastVis, f'HAI contrast {LAST} (China - Russia)')
m2.addLayer(paint(f'd_{FIRST}'), contrastVis, f'HAI contrast {FIRST}', False)
m2.addLayer(ee.Image().float().paint(cells.map(lambda f: f.set('dc', ee.Number(f.get(f'd_{LAST}')).subtract(f.get(f'd_{FIRST}')))), 'dc'),
            contrastVis, f'Change in contrast {FIRST} -> {LAST}', False)
m2.addLayer(borderLine, {'palette': 'black'}, 'Border')
m2

In [ ]:
# --- Hotspot zooms (paper Fig. 7 analogue): HAI>4 in first vs last year ----------------------------
def hotspot_map(lon, lat, zoom=10):
    mm = base_map((lat, lon), zoom)
    mm.addLayer(hai(FIRST).gt(HIGH_HAI).selfMask(), {'palette': ['#f2a900']}, f'HAI>4 {FIRST}')
    mm.addLayer(hai(LAST).gt(HIGH_HAI).selfMask(), {'palette': ['#7b2cbf']}, f'HAI>4 {LAST}')
    mm.addLayer(borderLine, {'palette': 'cyan'}, 'Border')
    return mm

seen = []
for _, r in top.iterrows():
    if all(km(r.lon, r.lat, a, b) > 80 for a, b in seen):     # skip near-duplicates
        seen.append((r.lon, r.lat))
        print(f'{r.near}  ({r.lon:.2f}, {r.lat:.2f})  contrast {FIRST}: {r[f"d_{FIRST}"]:+.2f}  {LAST}: {r[f"d_{LAST}"]:+.2f}  [{r[f"type_{LAST}"]}]')
        display(hotspot_map(r.lon, r.lat))
    if len(seen) == 4:
        break

**Reading the output**
* *Contrast* is the mean HAI of the Chinese part of a ~20 km cell minus the Russian part, on the 0–10 scale, so red/positive = China more active and blue/negative = Russia more active.
* *Hotspot type* follows the paper's idea (Fig. 7): HAI>4 area on one side only is **unilateral expansion**, on both sides is **bilateral expansion or cross-border integration** (twin cities such as Heihe–Blagoveshchensk). This notebook does not separate those last two.
* Town labels are the nearest entry in a short hand-written list (`PLACES`), not a gazetteer lookup.